# Module 8: Agentic AI with CrewAI, AG2/AutoGen, and LangGraph
## Topics Covered
- AI Workflow Design Patterns
- Multi-Agent Orchestration
- CrewAI — Role-based agent teams
- AG2 / AutoGen — Conversation-driven multi-agent systems
- LangGraph Human-in-the-Loop
- Multi-Agent Collaboration patterns

**Real-world scenario**: A content production pipeline where multiple specialized agents collaborate — researcher, writer, editor, SEO analyst — to produce a polished blog post.

In [ ]:
%pip install -q crewai crewai-tools langchain-openai
%pip install -q ag2
%pip install -q langgraph

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-api-key-here')
print('Ready:', os.environ['OPENAI_API_KEY'] != 'your-api-key-here')

---
## 1. Multi-Agent Orchestration Patterns

| Pattern | Framework | Best For |
|---------|-----------|----------|
| Role-based crews | CrewAI | Structured pipelines with defined roles |
| Conversation-driven | AG2/AutoGen | Dynamic back-and-forth agent dialogue |
| Graph-based state | LangGraph | Complex branching, loops, human-in-the-loop |

```
CrewAI:                    AG2:
  Manager                    Agent A <-> Agent B
  |-- Agent 1 (Researcher)       v          v
  |-- Agent 2 (Writer)       Agent C <-> Agent D
  |-- Agent 3 (Editor)      (any-to-any conversation)
  (sequential or hierarchical)
```

---
## 2. CrewAI — Role-Based Agent Teams

In [ ]:
from crewai import Agent, Task, Crew, Process
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7)

# Specialized agents with distinct roles and backstories
researcher = Agent(
    role='Senior Research Analyst',
    goal='Research {topic} and gather key facts, statistics, and insights',
    backstory='Expert researcher with 10 years experience. Focuses on accuracy and credible sources.',
    verbose=True, llm=llm, allow_delegation=False
)

writer = Agent(
    role='Content Writer',
    goal='Write engaging blog content on {topic} based on the research provided',
    backstory='Skilled content writer who transforms research into compelling, accessible articles for technical audiences.',
    verbose=True, llm=llm, allow_delegation=False
)

editor = Agent(
    role='Senior Editor',
    goal='Edit and polish the blog content on {topic} for clarity and impact',
    backstory='Meticulous editor who improves flow, fixes inconsistencies, and ensures publication standards.',
    verbose=True, llm=llm, allow_delegation=False
)

seo_analyst = Agent(
    role='SEO Analyst',
    goal='Optimize content on {topic} for search engines and add metadata',
    backstory='SEO expert who identifies best keywords, optimizes headings, and crafts meta descriptions.',
    verbose=True, llm=llm, allow_delegation=False
)

print('4 agents created: Researcher, Writer, Editor, SEO Analyst')

In [ ]:
# Tasks form the pipeline — each depends on previous output
research_task = Task(
    description='Research {topic}. Produce a brief covering: current state, top 5 facts/stats, key challenges, case studies, future trends.',
    expected_output='Structured research brief with facts, stats, and trends about {topic}',
    agent=researcher
)

writing_task = Task(
    description='Using the research brief, write a 600-800 word blog post on {topic} with: headline, intro, 3-4 sections, practical takeaways, CTA conclusion.',
    expected_output='Well-structured blog post of 600-800 words about {topic}',
    agent=writer,
    context=[research_task]
)

editing_task = Task(
    description='Edit the blog post on {topic}: improve clarity and flow, fix inconsistencies, strengthen intro and conclusion. Return complete edited article.',
    expected_output='Polished, publication-ready blog post about {topic}',
    agent=editor,
    context=[writing_task]
)

seo_task = Task(
    description='Optimize the blog post on {topic} for SEO: add meta description (150-160 chars), suggest primary/secondary keywords, optimize headline, add URL slug. Return final post with SEO metadata section at top.',
    expected_output='Final SEO-optimized blog post with metadata header for {topic}',
    agent=seo_analyst,
    context=[editing_task]
)

# Assemble the crew
content_crew = Crew(
    agents=[researcher, writer, editor, seo_analyst],
    tasks=[research_task, writing_task, editing_task, seo_task],
    process=Process.sequential,
    verbose=True
)

result = content_crew.kickoff(inputs={'topic': 'Retrieval Augmented Generation in Enterprise AI'})
print('\n' + '='*70)
print('FINAL CONTENT:')
print('='*70)
print(result.raw)

---
## 3. AG2 / AutoGen — Conversation-Driven Agents

In [ ]:
# AG2 is community continuation of Microsoft AutoGen
# Both ag2 and pyautogen share the same API
try:
    import autogen
except ImportError:
    import ag2 as autogen

llm_config = {
    'model': 'gpt-4o-mini',
    'api_key': os.environ['OPENAI_API_KEY'],
    'temperature': 0.3,
}

# Two-agent conversation: human proxy + AI assistant
human_proxy = autogen.UserProxyAgent(
    name='Human_Proxy',
    human_input_mode='NEVER',
    max_consecutive_auto_reply=4,
    code_execution_config={
        'work_dir': '/tmp/autogen_work',
        'use_docker': False
    },
    is_termination_msg=lambda x: 'TERMINATE' in x.get('content', '')
)

assistant = autogen.AssistantAgent(
    name='AI_Assistant',
    llm_config={'config_list': [llm_config]},
    system_message='''You are a helpful AI assistant. For analytical tasks:
1. Reason through the problem step by step
2. Write Python code when calculations are needed
3. Explain your results clearly
Say TERMINATE when the task is fully complete.'''
)

# Start a conversation to solve a business problem
human_proxy.initiate_chat(
    assistant,
    message='Calculate ROI: We invested $50,000 in AI automation. It saves 200 hours/month at $75/hr labor cost. Show ROI after 6 and 12 months, and the break-even point.',
    max_turns=4
)

In [ ]:
# AG2 Group Chat: multiple specialist agents collaborate
import os
os.makedirs('/tmp/autogen_work', exist_ok=True)

architect = autogen.AssistantAgent(
    name='Software_Architect',
    llm_config={'config_list': [llm_config]},
    system_message='You review code for design patterns, scalability, and architecture. Be concise.'
)

security = autogen.AssistantAgent(
    name='Security_Expert',
    llm_config={'config_list': [llm_config]},
    system_message='You review code for vulnerabilities, injection risks, and security best practices. Be concise.'
)

perf = autogen.AssistantAgent(
    name='Performance_Expert',
    llm_config={'config_list': [llm_config]},
    system_message='You review code for bottlenecks and optimization opportunities. Be concise.'
)

dev_proxy = autogen.UserProxyAgent(
    name='Developer',
    human_input_mode='NEVER',
    max_consecutive_auto_reply=0,
    code_execution_config=False
)

groupchat = autogen.GroupChat(
    agents=[dev_proxy, architect, security, perf],
    messages=[],
    max_round=5,
    speaker_selection_method='round_robin'
)

manager = autogen.GroupChatManager(
    groupchat=groupchat,
    llm_config={'config_list': [llm_config]}
)

code_snippet = '''
def get_user(user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    return db.execute(query)[0]
'''

dev_proxy.initiate_chat(
    manager,
    message=f'Review this Python function and give one key feedback point each:\n```python{code_snippet}```'
)

---
## 4. LangGraph: Human-in-the-Loop with Interrupts

In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

class ApprovalState(TypedDict):
    request: str
    draft_action: Optional[str]
    human_approved: Optional[bool]
    final_result: Optional[str]

def plan_action(state: ApprovalState) -> ApprovalState:
    prompt = f"""Request: {state['request']}
Describe the specific action you will take and its impact.
Format: ACTION: <what> | IMPACT: <effect>"""
    result = llm.invoke([HumanMessage(content=prompt)])
    return {**state, 'draft_action': result.content}

def execute_approved(state: ApprovalState) -> ApprovalState:
    result = llm.invoke([HumanMessage(content=f'Confirm execution of: {state["draft_action"]}')])
    return {**state, 'final_result': f'COMPLETED: {result.content}'}

def reject(state: ApprovalState) -> ApprovalState:
    return {**state, 'final_result': 'REJECTED: Action cancelled by human reviewer. No changes made.'}

def route_approval(state: ApprovalState) -> str:
    if state.get('human_approved') is True:  return 'execute'
    if state.get('human_approved') is False: return 'reject'
    return END  # Interrupt — wait for human input

g = StateGraph(ApprovalState)
g.add_node('plan', plan_action)
g.add_node('execute', execute_approved)
g.add_node('reject', reject)
g.add_edge(START, 'plan')
g.add_conditional_edges('plan', route_approval, {'execute':'execute','reject':'reject', END:END})
g.add_edge('execute', END)
g.add_edge('reject', END)

memory = MemorySaver()
app = g.compile(checkpointer=memory, interrupt_after=['plan'])

cfg = {'configurable': {'thread_id': 'approval-001'}}

# Step 1: Plan (pauses here for human review)
s = app.invoke({'request': 'Delete all inactive user accounts from the database', 'draft_action': None, 'human_approved': None, 'final_result': None}, config=cfg)
print('=== AWAITING HUMAN APPROVAL ===')
print(f'Proposed Action:\n{s["draft_action"]}')

In [ ]:
# Step 2: Human rejects the action
app.update_state(cfg, {'human_approved': False})
final = app.invoke(None, config=cfg)
print(f'Result: {final["final_result"]}')

# New safe action — human approves
cfg2 = {'configurable': {'thread_id': 'approval-002'}}
s2 = app.invoke({'request': 'Send a monthly digest email to newsletter subscribers', 'draft_action': None, 'human_approved': None, 'final_result': None}, config=cfg2)
print(f'\nProposed: {s2["draft_action"]}')

app.update_state(cfg2, {'human_approved': True})
final2 = app.invoke(None, config=cfg2)
print(f'\nResult: {final2["final_result"]}')

---
## 5. Framework Comparison

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    'Feature': ['Agent communication','State management','Workflow type','Code execution','Human-in-the-loop','Learning curve','Production readiness'],
    'CrewAI': ['Task-based handoff','Crew-level','Sequential/Hierarchical','Via tools','Limited','Low','High'],
    'AG2/AutoGen': ['Conversational messages','Per-conversation','Conversation-driven','Built-in sandbox','UserProxyAgent','Medium','Medium'],
    'LangGraph': ['Graph state transitions','Typed state dict','Any graph + cycles','Via tool nodes','interrupt_after','High','High']
})

print('Framework Comparison:')
print(comparison.to_string(index=False))

print('\n\nWhen to use each:')
print('  CrewAI    → Clear roles, structured pipeline (content gen, data processing)')
print('  AG2       → Agents need to debate, reason together, or execute dynamic code')
print('  LangGraph → Fine-grained control, approval gates, complex branching')

---
## Course Complete!

| Module | Topic |
|--------|-------|
| 1 | Generative AI: Prompting, Templates, LangChain LCEL |
| 2 | RAG: LangChain pipelines, LlamaIndex, Gradio UI |
| 3 | Vector DBs: ChromaDB, embeddings, recommendation systems |
| 4 | Advanced RAG: FAISS, hybrid retrieval, compression |
| 5 | Multimodal AI: Vision, DALL-E, Whisper, TTS |
| 6 | AI Agents: Function calling, DataFrame/SQL agents |
| 7 | LangGraph: Stateful workflows, ReAct, multi-agent graphs |
| 8 | Multi-agent: CrewAI, AG2/AutoGen, human-in-the-loop |